<!--
Copyright Amazon.com, Inc. or its affiliates. All Rights Reserved.
SPDX-License-Identifier: MIT-0
-->


In [ ]:
%store -r

In [ ]:
%store

# Lab 1: Data Preparation

## What this lab does, and why it matters

Every text-to-SQL tutorial needs a dataset of question/query pairs. Most hand you one. This lab **builds yours from the query log of a real database.**

The Aurora cluster CloudFormation deployed for you has already had roughly 250 analytical queries replayed against it. PostgreSQL recorded every one in `pg_stat_statements`, an extension that tracks aggregated execution statistics per statement. Those queries — with real call counts and real execution times — are the raw material. What this lab does not have is the *other* half of a training pair: nobody wrote down what question each query was answering. So we generate that with Amazon Bedrock, then verify our work by executing each candidate query against the live database.

The result is two datasets in two different shapes:

| Format | Shape | Consumed by |
|---|---|---|
| **SFT** | `{"prompt": "...", "completion": "..."}` | Lab 2 — supervised fine-tuning |
| **RLVR** | conversation messages + `reward_model.ground_truth` | Lab 3 — reinforcement learning |

### Why this is the realistic workflow

On a real project your query log is exactly where you would start. It is almost always more representative of what your users actually ask than anything you would write by hand — including the awkward queries, the ones with six-way joins, and the ones people run forty times a day. Synthetic datasets tend to be tidier than reality, and a model fine-tuned on tidy data is confident and wrong on the messy parts.

### How to run this notebook

Cells 3-11 define the pipeline's building blocks — helper functions, one group per cell. Nothing executes against the database until **Step 1** further down.

From there the pipeline runs in **seven explicit steps**, each with a preview cell showing you what it produced. Those previews are the point: you should finish this lab having *seen* your own training data, not just a summary count of it.

::alert[Step 2 calls Amazon Bedrock once or twice per query and executes each candidate against Aurora, so it takes roughly 10 minutes. That is the slowest part of the lab. Start it and read ahead.]{type="info"}


## Pipeline building blocks

The next nine cells define the functions the pipeline uses. They are grouped by
job — configuration, database access, extraction, cleaning, formatting,
splitting, quality checks, schema. **Run them all; none of them do anything on
their own.** Execution starts at Step 1.


In [ ]:
import json
import os
import re
import hashlib
import logging
from typing import Optional
import boto3


### Configuration

Paths, split ratio, and the query-length bounds used to filter out trivial or
pathological queries. `MIN_SAMPLES_REQUIRED = 5` is a floor that stops the lab
early with a clear message rather than letting you fine-tune on nothing.

The Aurora ARNs come from `%store -r` — Lab 0 put them there. If this cell
raises `NameError`, run `00-setup.ipynb` first.


In [ ]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
logger = logging.getLogger(__name__)

%store -r

# Confirm Lab 0 populated the Aurora identifiers before we use them. Without
# this guard, a fresh kernel (or a skipped Lab 0) fails a few lines down with a
# bare "NameError: AURORA_CLUSTER_ARN" that gives no hint at the cause. Fail
# early with an actionable message instead.
_required = ["AURORA_CLUSTER_ARN", "AURORA_SECRET_ARN", "AURORA_DB_NAME"]
_missing = [v for v in _required if not globals().get(v)]
if _missing:
    raise RuntimeError(
        f"Missing/empty variables from Lab 0: {', '.join(_missing)}. "
        "Run 00-setup.ipynb to completion first (it %store's these), then "
        "re-run this cell. If Lab 0 was already run, your kernel restarted -- "
        "re-running 00-setup.ipynb restores them."
    )
print(f"Aurora config restored from Lab 0:")
print(f"  Cluster: {AURORA_CLUSTER_ARN}")
print(f"  Secret:  {AURORA_SECRET_ARN}")
print(f"  DB:      {AURORA_DB_NAME}")

# Set environment variables for the pipeline script
os.environ["AURORA_CLUSTER_ARN"] = AURORA_CLUSTER_ARN
os.environ["AURORA_SECRET_ARN"] = AURORA_SECRET_ARN
os.environ["AURORA_DB_NAME"] = AURORA_DB_NAME

# Output paths
OUTPUT_DIR = os.environ.get("OUTPUT_DIR", ".")
TRAIN_OUTPUT = os.path.join(OUTPUT_DIR, "train.jsonl")
VALIDATION_OUTPUT = os.path.join(OUTPUT_DIR, "validation.jsonl")
COMBINED_OUTPUT = os.path.join(OUTPUT_DIR, "combined.jsonl")
RLVR_DIR = os.path.join(OUTPUT_DIR, "rlvr_data")
RLVR_TRAIN_OUTPUT = os.path.join(RLVR_DIR, "rl_train.jsonl")
RLVR_VAL_OUTPUT = os.path.join(RLVR_DIR, "rl_val.jsonl")
RLVR_COMBINED_OUTPUT = os.path.join(RLVR_DIR, "rl_combined.jsonl")
RAW_EXPORT = os.path.join(OUTPUT_DIR, "extracted_queries_raw.json")

# Pipeline parameters
TRAIN_SPLIT_RATIO = 0.85
MIN_QUERY_LENGTH = 15
MAX_QUERY_LENGTH = 2000
MIN_SAMPLES_REQUIRED = 5


# Helper used by the preview cells further down to keep long SQL and schema
# blocks readable in the notebook output.
def _short(text, limit=400):
    text = str(text)
    return text if len(text) <= limit else text[:limit] + f"\n  ... [{len(text) - limit} more chars]"


### Aurora access via the Data API

`boto3.client("rds-data")` talks to Aurora over HTTPS, so the notebook needs no
VPC connection, no Postgres driver, and no password — credentials come from the
Secrets Manager ARN. This same mechanism is what makes the Lab 2 reward function
possible: scoring generated SQL means *running* it, and here that is one API
call.


In [ ]:
# ---------------------------------------------------------------------------
# Aurora Data API Client
# ---------------------------------------------------------------------------


def get_aurora_data_client():
    """Create an Aurora Data API client."""
    region = AURORA_CLUSTER_ARN.split(":")[3] if AURORA_CLUSTER_ARN else None
    return boto3.client("rds-data", region_name=region)


def execute_statement(sql: str, parameters: list = None) -> dict:
    """
    Execute a SQL statement via the Aurora Data API.

    No VPC connectivity or database drivers required - uses HTTP.
    """
    if not AURORA_CLUSTER_ARN:
        raise RuntimeError(
            "AURORA_CLUSTER_ARN not set. Export it from the CloudFormation stack output:\n"
            "  export AURORA_CLUSTER_ARN=$(aws cloudformation describe-stacks "
            "--stack-name query-training-aurora "
            "--query 'Stacks[0].Outputs[?OutputKey==`ClusterARN`].OutputValue' --output text)"
        )
    if not AURORA_SECRET_ARN:
        raise RuntimeError(
            "AURORA_SECRET_ARN not set. Export it from the CloudFormation stack output:\n"
            "  export AURORA_SECRET_ARN=$(aws cloudformation describe-stacks "
            "--stack-name query-training-aurora "
            "--query 'Stacks[0].Outputs[?OutputKey==`SecretARN`].OutputValue' --output text)"
        )

    client = get_aurora_data_client()

    kwargs = {
        "resourceArn": AURORA_CLUSTER_ARN,
        "secretArn": AURORA_SECRET_ARN,
        "database": AURORA_DB_NAME,
        "sql": sql,
        "includeResultMetadata": True,
    }
    if parameters:
        kwargs["parameters"] = parameters

    return client.execute_statement(**kwargs)


def parse_data_api_response(response: dict) -> list[dict]:
    """
    Parse Aurora Data API response into a list of dictionaries.

    The Data API returns column metadata and records separately;
    this combines them into familiar row-dict format.
    """
    columns = [col["name"] for col in response.get("columnMetadata", [])]
    records = response.get("records", [])

    rows = []
    for record in records:
        row = {}
        for i, field in enumerate(record):
            col_name = columns[i] if i < len(columns) else f"col_{i}"
            # Data API returns typed values like {"stringValue": "..."} or {"longValue": 123}
            if "stringValue" in field:
                row[col_name] = field["stringValue"]
            elif "longValue" in field:
                row[col_name] = field["longValue"]
            elif "doubleValue" in field:
                row[col_name] = field["doubleValue"]
            elif "booleanValue" in field:
                row[col_name] = field["booleanValue"]
            elif "isNull" in field and field["isNull"]:
                row[col_name] = None
            else:
                row[col_name] = str(field)
        rows.append(row)

    return rows

### Query extraction

The `EXTRACTION_QUERY` reads `pg_stat_statements`, joining to `pg_database` and
`pg_roles` for context. Note what it filters on — `SELECT` statements only,
excluding the system catalogs and our own extraction queries, ordered by call
count. The most-run queries are the most representative ones.


In [ ]:
# ---------------------------------------------------------------------------
# Query Extraction from pg_stat_statements
# ---------------------------------------------------------------------------

EXTRACTION_QUERY = """
SELECT
    pss.query,
    pss.calls,
    pss.mean_exec_time,
    pss.rows AS avg_rows_returned,
    d.datname AS database_name,
    r.rolname AS user_name
FROM pg_stat_statements pss
JOIN pg_database d ON d.oid = pss.dbid
JOIN pg_roles r ON r.oid = pss.userid
WHERE
    pss.query ~* '^\\s*(SELECT|WITH)'
    AND pss.query ILIKE '%product_sales%'
    AND d.datname = 'querytraining'
    AND pss.query NOT ILIKE '%pg_%'
    AND pss.query NOT ILIKE '%information_schema%'
    AND pss.query NOT ILIKE '%pg_catalog%'
    AND pss.query NOT ILIKE '%rds_%'
    AND LENGTH(pss.query) > :min_length
    AND LENGTH(pss.query) < :max_length
    AND pss.calls >= 1
ORDER BY pss.calls DESC, pss.mean_exec_time ASC
LIMIT 5000;
"""


def extract_queries_from_aurora() -> list[dict]:
    """Extract user queries from Aurora pg_stat_statements via Data API."""
    logger.info("Querying Aurora PostgreSQL via Data API...")

    parameters = [
        {"name": "min_length", "value": {"longValue": MIN_QUERY_LENGTH}},
        {"name": "max_length", "value": {"longValue": MAX_QUERY_LENGTH}},
    ]

    try:
        response = execute_statement(EXTRACTION_QUERY, parameters)
        rows = parse_data_api_response(response)
        logger.info(f"Extracted {len(rows)} raw queries from Aurora cluster.")
        return rows
    except Exception as e:
        if "UndefinedTable" in str(e) or "pg_stat_statements" in str(e):
            logger.error(
                "pg_stat_statements not found. Creating extension...\n"
                "Run: CREATE EXTENSION IF NOT EXISTS pg_stat_statements;"
            )
            # Attempt to create it
            try:
                execute_statement("CREATE EXTENSION IF NOT EXISTS pg_stat_statements;")
                logger.info("Extension created. Re-running extraction...")
                response = execute_statement(EXTRACTION_QUERY, parameters)
                rows = parse_data_api_response(response)
                logger.info(f"Extracted {len(rows)} raw queries from Aurora cluster.")
                return rows
            except Exception as inner_e:
                logger.error(f"Failed to create extension: {inner_e}")
                raise
        raise


### Cleaning and description generation

The largest cell in the notebook, and the interesting one. For each raw query it:

1. Normalises whitespace and strips the `$1`, `$2` parameter placeholders that
   `pg_stat_statements` leaves behind, substituting plausible literal values.
2. **Executes the result against Aurora** to confirm it is still valid SQL.
3. If execution fails, sends the error back to Bedrock and asks for a repair —
   then re-executes. One retry, not a loop.
4. Asks Bedrock to write the natural-language question the query answers
   (`generate_natural_language_description`), falling back to a template-based
   `_heuristic_description` if Bedrock is unavailable.

Anything that cannot be made to execute is dropped. That is why the count after
cleaning is lower than the count extracted — and it is a feature: an unverifiable
pair would teach the model to write SQL that does not run.


In [ ]:
# ---------------------------------------------------------------------------
# Data Cleaning
# ---------------------------------------------------------------------------


def normalize_whitespace(text: str) -> str:
    """Collapse multiple whitespace/newlines into single spaces."""
    return re.sub(r"\s+", " ", text).strip()


def remove_special_characters(text: str) -> str:
    """Remove non-printable and control characters, keep SQL-valid chars."""
    cleaned = re.sub(r"[^\x20-\x7E]", " ", text)
    return normalize_whitespace(cleaned)


def remove_parameter_placeholders(query: str) -> str:
    """
    Replace $1, $2, ... placeholders from pg_stat_statements with literal values.

    Uses Bedrock (Claude) to infer contextually appropriate literal replacements
    based on the query structure, column names, and SQL patterns.
    """
    if not re.search(r"\$\d+", query):
        return query

    try:
        bedrock = boto3.client("bedrock-runtime", region_name=AURORA_CLUSTER_ARN.split(":")[3])

        prompt = (
            "You are a SQL expert. The following PostgreSQL query has parameterized "
            "placeholders ($1, $2, etc.) from pg_stat_statements. Replace each placeholder "
            "with a realistic literal value that makes sense given the column names, table "
            "structure, and query context.\n\n"
            "Rules:\n"
            "- Use string literals in single quotes for text columns\n"
            "- Use numeric literals (no quotes) for numbers\n"
            "- Use date literals like '2025-06-01' for date columns\n"
            "- For LIMIT placeholders, use 10\n"
            "- For price thresholds, use values like 20 or 100\n"
            "- For country_code, use real 2-letter ISO codes like 'FR'\n"
            "- For product_category, use values like 'hair_care', 'fragrance', etc.\n"
            "- For brand_tier, use 'luxury', 'premium', 'mass_market', or 'indie'\n"
            "- Return ONLY the modified SQL query, nothing else\n\n"
            f"Query:\n{query}"
        )

        response = bedrock.invoke_model(
            modelId="us.anthropic.claude-sonnet-4-6",
            contentType="application/json",
            accept="application/json",
            body=json.dumps({
                "anthropic_version": "bedrock-2023-05-31",
                "max_tokens": 1024,
                "messages": [{"role": "user", "content": prompt}],
            }),
        )

        result = json.loads(response["body"].read())
        replaced_query = result["content"][0]["text"].strip()

        if is_valid_sql(replaced_query) and not re.search(r"\$\d+", replaced_query):
            logger.debug(f"Replaced placeholders: {query[:60]}... -> {replaced_query[:60]}...")
            return replaced_query
        else:
            logger.warning(f"Bedrock replacement still has placeholders or is invalid, skipping: {query[:80]}")
            return query

    except Exception as e:
        logger.warning(f"Bedrock placeholder replacement failed: {e}. Keeping original query.")
        return query


def is_valid_sql(query: str) -> bool:
    """Basic validation that the query looks like valid SQL."""
    query_upper = query.upper().strip()

    # Must start with SELECT or WITH
    if not re.match(r"^\s*(SELECT|WITH)\b", query_upper):
        return False

    # Must contain at least a FROM clause (except for simple SELECT expressions)
    has_from = "FROM" in query_upper
    is_expression = re.match(r"^\s*SELECT\s+\d", query_upper)
    if not has_from and not is_expression:
        return False

    # Check for balanced parentheses
    if query.count("(") != query.count(")"):
        return False

    # Reject if it's mostly placeholders
    placeholder_count = len(re.findall(r"\$\d+", query))
    word_count = len(query.split())
    if word_count > 0 and placeholder_count / word_count > 0.5:
        return False

    return True


def generate_natural_language_description(sql_query: str) -> str:
    """
    Generate a natural language description for a SQL query using Bedrock.

    Produces descriptions that sound like a business user exploring data,
    not a DBA writing pseudocode.
    """
    try:
        bedrock = boto3.client("bedrock-runtime", region_name=AURORA_CLUSTER_ARN.split(":")[3])

        prompt = (
            "You are helping generate training data for a text-to-SQL model. "
            "Given the SQL query below, write a short natural language question that "
            "a business user or analyst would ask to get this data.\n\n"
            "Rules:\n"
            "- Write as if you're a non-technical person exploring cosmetics sales data\n"
            "- Use plain conversational English, not technical jargon\n"
            "- Don't mention SQL, tables, columns, GROUP BY, or any database concepts\n"
            "- Use business terms: 'revenue' not 'SUM(price * quantity)', 'countries' not 'country_code'\n"
            "- Keep it to one short sentence (under 15 words ideally)\n"
            "- Don't start with 'Show me' every time — vary the phrasing\n"
            "- Return ONLY the natural language question, nothing else\n\n"
            f"SQL:\n{sql_query}"
        )

        response = bedrock.invoke_model(
            modelId="us.anthropic.claude-sonnet-4-6",
            contentType="application/json",
            accept="application/json",
            body=json.dumps({
                "anthropic_version": "bedrock-2023-05-31",
                "max_tokens": 100,
                "messages": [{"role": "user", "content": prompt}],
            }),
        )

        result = json.loads(response["body"].read())
        description = result["content"][0]["text"].strip().strip('"')

        if len(description) >= 5:
            return description

    except Exception as e:
        logger.warning(f"Bedrock NL generation failed: {e}. Falling back to heuristic.")

    return _heuristic_description(sql_query)


def _heuristic_description(sql_query: str) -> str:
    """Fallback heuristic description if Bedrock is unavailable."""
    query_upper = sql_query.upper()

    from_match = re.search(r"FROM\s+(\w+)", query_upper)
    table_name = from_match.group(1).lower() if from_match else "table"

    if "GROUP BY" in query_upper:
        return f"Break down {table_name} data by category"
    elif "ORDER BY" in query_upper and "LIMIT" in query_upper:
        return f"What are the top results from {table_name}?"
    elif "WHERE" in query_upper:
        return f"Filter {table_name} data by specific criteria"
    else:
        return f"Query {table_name} data"


def validate_query_against_db(query: str) -> tuple[bool, str]:
    """
    Execute a query against the database to confirm it runs successfully.
    Returns (success, error_message).
    """
    try:
        execute_statement(query)
        return True, ""
    except Exception as e:
        return False, str(e)


def attempt_query_fix(query: str, error_msg: str) -> Optional[str]:
    """
    Use Sonnet to fix a query that failed execution, based on the error message.
    Returns the fixed query or None if the fix attempt fails.
    """
    try:
        bedrock = boto3.client("bedrock-runtime", region_name=AURORA_CLUSTER_ARN.split(":")[3])

        prompt = (
            "You are a PostgreSQL expert. The following SQL query failed when executed "
            "against an Aurora PostgreSQL database. Fix the query so it executes successfully.\n\n"
            "Rules:\n"
            "- The table is product_sales with columns: sale_id, product_name, product_category, "
            "brand, brand_tier, price, volume_ml, quantity, country_code, sale_date, launch_date, "
            "customer_segment, target_age_group, certification_tags, ingredients_type, season_tag, "
            "rating, stock_status, sales_rank\n"
            "- Use PostgreSQL syntax (TO_CHAR not DATE_FORMAT, EXTRACT not YEAR(), "
            "CURRENT_DATE - INTERVAL not DATE_SUB, etc.)\n"
            "- In HAVING clauses, repeat the aggregate expression instead of using aliases\n"
            "- All columns in ORDER BY must appear in GROUP BY or be aggregated\n"
            "- Return ONLY the fixed SQL query, nothing else\n\n"
            f"Failed query:\n{query}\n\n"
            f"Error:\n{error_msg}"
        )

        response = bedrock.invoke_model(
            modelId="us.anthropic.claude-sonnet-4-6",
            contentType="application/json",
            accept="application/json",
            body=json.dumps({
                "anthropic_version": "bedrock-2023-05-31",
                "max_tokens": 1024,
                "messages": [{"role": "user", "content": prompt}],
            }),
        )

        result = json.loads(response["body"].read())
        fixed_query = result["content"][0]["text"].strip()

        # Strip markdown code fencing if present
        fixed_query = re.sub(r"^```sql\s*", "", fixed_query)
        fixed_query = re.sub(r"\s*```$", "", fixed_query)
        fixed_query = fixed_query.strip()

        if is_valid_sql(fixed_query) and "product_sales" in fixed_query.lower():
            return fixed_query

    except Exception as e:
        logger.warning(f"Query fix attempt failed: {e}")

    return None


def clean_single_query(raw: dict) -> Optional[dict]:
    """Clean and validate a single extracted query record."""
    query = raw.get("query", "")
    if not query:
        return None

    # Normalize whitespace and remove special chars
    query = remove_special_characters(query)
    query = remove_parameter_placeholders(query)

    # Only retain queries targeting the product_sales table
    if "product_sales" not in query.lower():
        return None

    # Validate structure
    if not is_valid_sql(query):
        return None

    if len(query) < MIN_QUERY_LENGTH or len(query) > MAX_QUERY_LENGTH:
        return None

    # Execute against the database to confirm it works
    success, error_msg = validate_query_against_db(query)

    if not success:
        logger.info(f"Query failed execution, attempting fix: {query[:80]}...")
        fixed_query = attempt_query_fix(query, error_msg)

        if fixed_query:
            retry_success, retry_error = validate_query_against_db(fixed_query)
            if retry_success:
                logger.info(f"Fix succeeded: {fixed_query[:80]}...")
                query = fixed_query
            else:
                logger.info(f"Fix still failed, rejecting: {retry_error[:80]}")
                return None
        else:
            logger.info("No fix produced, rejecting query.")
            return None

    # Generate a natural language description
    user_query = generate_natural_language_description(query)
    if not user_query or len(user_query) < 5:
        return None

    return {
        "user_query": user_query,
        "expected_sql": query,
        "calls": raw.get("calls", 0),
        "mean_exec_time_ms": round(float(raw.get("mean_exec_time", 0)), 2),
        "database_name": raw.get("database_name", ""),
        "user_name": raw.get("user_name", ""),
    }


def clean_dataset(raw_queries: list[dict]) -> list[dict]:
    """Clean the full dataset: validate, deduplicate, and filter."""
    logger.info(f"Cleaning {len(raw_queries)} raw queries...")

    cleaned = []
    seen_hashes = set()
    rejected_reasons = {
        "invalid_sql": 0,
        "duplicate": 0,
        "too_short_description": 0,
        "cleaning_failed": 0,
    }

    for raw in raw_queries:
        result = clean_single_query(raw)
        if result is None:
            rejected_reasons["cleaning_failed"] += 1
            continue

        # Deduplicate by SQL content hash
        sql_hash = hashlib.md5(
            result["expected_sql"].lower().encode()
        ).hexdigest()

        if sql_hash in seen_hashes:
            rejected_reasons["duplicate"] += 1
            continue

        seen_hashes.add(sql_hash)
        cleaned.append(result)

    logger.info(f"Cleaning complete: {len(cleaned)} valid samples retained.")
    logger.info(f"Rejection breakdown: {json.dumps(rejected_reasons, indent=2)}")
    return cleaned


### Formatting for two different trainers

`format_for_training` produces the SFT shape: a single instruction prompt string
and the SQL as the completion.

`format_for_rlvr` produces the VERL shape RLVR expects, which is structurally
different — the prompt is a **list of chat messages**, and the target SQL lives
under `reward_model.ground_truth` rather than in a `completion` field. That is
not arbitrary: in RLVR the model generates its own candidates and the ground
truth is only ever handed to the *reward function*, never shown to the model. The
data layout reflects who is allowed to see what.

Both share `schema_context`, which is why the schema extraction step exists.


In [ ]:
# ---------------------------------------------------------------------------
# Formatting to Training JSONL
# ---------------------------------------------------------------------------


def build_training_prompt(user_query: str, schema_context: str = "") -> str:
    """Build the instruction prompt for fine-tuning."""
    prompt = f"""### Instruction:
You are a SQL query generator for a PostgreSQL database.

{schema_context}
Generate only the SQL query without explanation.

### User Query:
{user_query}
"""
    return prompt


def format_for_training(sample: dict, schema_context: str = "") -> dict:
    """Format a cleaned sample into the SageMaker AI SFT training format."""
    prompt = build_training_prompt(sample["user_query"], schema_context)
    return {
        "prompt": prompt,
        "completion": sample["expected_sql"],
    }


def format_for_rlvr(sample: dict, schema_context: str = "", index: int = 0) -> dict:
    """Format a cleaned sample into the SageMaker AI RLVR training format (VERL)."""
    system_content = (
        "You are a SQL query generator for a PostgreSQL database.\n\n"
        f"{schema_context}\n"
        "Generate only the SQL query without explanation."
    )
    return {
        "data_source": "custom/sql_generation",
        "prompt": [
            {"role": "system", "content": system_content},
            {"role": "user", "content": sample["user_query"]},
        ],
        "ability": "sql",
        "reward_model": {
            "style": "rule",
            "ground_truth": sample["expected_sql"],
        },
        "extra_info": {
            "split": "train",
            "index": index,
        },
    }

### Train/validation split

85/15, shuffled with `random.seed(42)`. The fixed seed is what makes the split
reproducible — and, as Step 6 explains, it is also what keeps the SFT and RLVR
splits aligned with each other.


In [ ]:
# ---------------------------------------------------------------------------
# Train/Validation Split
# ---------------------------------------------------------------------------


def split_dataset(
    samples: list[dict], train_ratio: float = TRAIN_SPLIT_RATIO
) -> tuple[list[dict], list[dict]]:
    """Split dataset into training and validation sets."""
    import random

    random.seed(42)
    shuffled = samples.copy()
    random.shuffle(shuffled)

    split_idx = int(len(shuffled) * train_ratio)
    train_set = shuffled[:split_idx]
    val_set = shuffled[split_idx:]

    logger.info(
        f"Split: {len(train_set)} training, {len(val_set)} validation "
        f"(ratio={train_ratio:.2f})"
    )
    return train_set, val_set


### Quality checks and validation

`run_quality_checks` profiles the finished dataset — average SQL and description
length, how many queries use `GROUP BY` / `JOIN` / `WHERE` / subqueries, and the
duplicate-description ratio. It flags issues rather than failing: a 30%+
duplicate rate means Bedrock produced generic descriptions and your model will
learn less than the sample count suggests.

`validate_jsonl_file` checks the **SFT** shape — that every line is valid JSON
with non-empty `prompt` and `completion`. Because the RLVR files have neither
field, Step 7 adds a sibling validator for them.


In [ ]:
# ---------------------------------------------------------------------------
# Quality Checks and Validation
# ---------------------------------------------------------------------------


def run_quality_checks(samples: list[dict]) -> dict:
    """Run quality checks on the final dataset."""
    logger.info("Running quality checks...")

    stats = {
        "total_samples": len(samples),
        "avg_query_length": 0,
        "avg_description_length": 0,
        "min_query_length": float("inf"),
        "max_query_length": 0,
        "has_select": 0,
        "has_group_by": 0,
        "has_join": 0,
        "has_where": 0,
        "has_subquery": 0,
        "unique_descriptions": set(),
        "issues": [],
    }

    for sample in samples:
        sql = sample.get("expected_sql", "")
        desc = sample.get("user_query", "")
        sql_upper = sql.upper()

        stats["avg_query_length"] += len(sql)
        stats["avg_description_length"] += len(desc)
        stats["min_query_length"] = min(stats["min_query_length"], len(sql))
        stats["max_query_length"] = max(stats["max_query_length"], len(sql))

        if "SELECT" in sql_upper:
            stats["has_select"] += 1
        if "GROUP BY" in sql_upper:
            stats["has_group_by"] += 1
        if "JOIN" in sql_upper:
            stats["has_join"] += 1
        if "WHERE" in sql_upper:
            stats["has_where"] += 1
        if sql_upper.count("SELECT") > 1:
            stats["has_subquery"] += 1

        stats["unique_descriptions"].add(desc.lower())

    n = max(len(samples), 1)
    stats["avg_query_length"] = round(stats["avg_query_length"] / n, 1)
    stats["avg_description_length"] = round(stats["avg_description_length"] / n, 1)
    stats["unique_description_count"] = len(stats["unique_descriptions"])
    del stats["unique_descriptions"]  # Not JSON serializable

    # Quality issue detection
    if stats["total_samples"] < MIN_SAMPLES_REQUIRED:
        stats["issues"].append(
            f"WARNING: Only {stats['total_samples']} samples, "
            f"minimum recommended is {MIN_SAMPLES_REQUIRED}"
        )

    duplicate_ratio = 1 - (stats["unique_description_count"] / max(n, 1))
    if duplicate_ratio > 0.3:
        stats["issues"].append(
            f"WARNING: {duplicate_ratio:.0%} of descriptions are duplicates"
        )

    if stats["has_select"] < n * 0.95:
        stats["issues"].append("WARNING: Some samples may not be valid SELECT queries")

    return stats


def validate_jsonl_file(filepath: str) -> bool:
    """Validate that a JSONL file is well-formed and complete."""
    logger.info(f"Validating {filepath}...")
    line_count = 0
    errors = []

    with open(filepath, "r") as f:
        for i, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                record = json.loads(line)
                if "prompt" not in record:
                    errors.append(f"Line {i}: missing 'prompt' field")
                if "completion" not in record:
                    errors.append(f"Line {i}: missing 'completion' field")
                if record.get("prompt", "").strip() == "":
                    errors.append(f"Line {i}: empty prompt")
                if record.get("completion", "").strip() == "":
                    errors.append(f"Line {i}: empty completion")
                line_count += 1
            except json.JSONDecodeError as e:
                errors.append(f"Line {i}: invalid JSON - {e}")

    if errors:
        for err in errors[:10]:
            logger.error(err)
        logger.error(f"Validation FAILED: {len(errors)} errors in {filepath}")
        return False

    logger.info(f"Validation PASSED: {line_count} valid records in {filepath}")
    return True

### Schema extraction

Reads `information_schema` and renders the table and column definitions as text.
This gets embedded in every training prompt so the model learns *this* schema by
name, rather than guessing column names — which is precisely the capability a
frontier model with no access to your database cannot have.


In [ ]:
# ---------------------------------------------------------------------------
# Schema Extraction (enriches prompts with real schema via Data API)
# ---------------------------------------------------------------------------


def extract_schema_context() -> str:
    """Extract table/column schema from Aurora for prompt enrichment."""
    schema_query = """
    SELECT
        t.table_schema,
        t.table_name,
        c.column_name,
        c.data_type,
        c.is_nullable
    FROM information_schema.tables t
    JOIN information_schema.columns c
        ON t.table_name = c.table_name
        AND t.table_schema = c.table_schema
    WHERE t.table_schema NOT IN ('pg_catalog', 'information_schema')
        AND t.table_type = 'BASE TABLE'
    ORDER BY t.table_schema, t.table_name, c.ordinal_position
    LIMIT 500;
    """
    try:
        response = execute_statement(schema_query)
        rows = parse_data_api_response(response)

        if not rows:
            return ""

        # Group by table
        tables = {}
        for row in rows:
            schema = row.get("table_schema", "public")
            table = row.get("table_name", "")
            column = row.get("column_name", "")
            dtype = row.get("data_type", "")
            nullable = row.get("is_nullable", "YES")

            full_name = f"{schema}.{table}" if schema != "public" else table
            if full_name not in tables:
                tables[full_name] = []
            null_str = "" if nullable == "YES" else " NOT NULL"
            tables[full_name].append(f"  - {column} ({dtype}{null_str})")

        # Format as schema context
        lines = ["DATABASE SCHEMA:"]
        for table_name, columns in tables.items():
            lines.append(f"\nTable: {table_name}")
            lines.extend(columns)

        return "\n".join(lines)

    except Exception as e:
        logger.warning(f"Could not extract schema context: {e}")
        return ""


---

# Step 1 of 7: Extract real queries from `pg_stat_statements`

Nothing has touched the database yet. This is the first call.

`extract_queries_from_aurora()` reads PostgreSQL's own record of every statement
it has executed. What comes back is not clean training data — placeholders
instead of literals, no descriptions, duplicates — but it is *real*: these are
queries that genuinely ran against this schema.

The raw export is written to `extracted_queries_raw.json` so you can come back to
it, and so you can see what the cleaning step in Step 2 actually had to work with.


In [ ]:
# [Step 1/7] Extract queries from Aurora via the Data API
logger.info("=" * 70)
logger.info("Aurora PostgreSQL Query Extraction Pipeline")
logger.info("=" * 70)
logger.info(f"  Cluster ARN: {AURORA_CLUSTER_ARN}")
logger.info(f"  Secret ARN:  {AURORA_SECRET_ARN}")
logger.info(f"  Database:    {AURORA_DB_NAME}")

logger.info("\n[Step 1/7] Extracting queries from Aurora pg_stat_statements...")
raw_queries = extract_queries_from_aurora()

if not raw_queries:
    raise RuntimeError(
        "No queries extracted. Check the cluster status and permissions, and "
        "ensure the pg_stat_statements extension is enabled:\n"
        "  CREATE EXTENSION IF NOT EXISTS pg_stat_statements;"
    )

# Save the raw export so this step is inspectable and re-runnable.
with open(RAW_EXPORT, "w") as f:
    json.dump(raw_queries, f, indent=2, default=str)

print(f"\nExtracted {len(raw_queries)} raw queries -> {RAW_EXPORT}")


In [ ]:
# Preview: what pg_stat_statements actually gave us.
#
# Look at `calls` and `mean_exec_time` — this is real usage data, not a
# synthetic dataset. Also note the $1/$2 placeholders in the SQL: Postgres
# normalises literals out of the statements it records, which is the first
# thing Step 2 has to undo.


print(f"{len(raw_queries)} raw queries extracted. First 2:\n")
for i, q in enumerate(raw_queries[:2], 1):
    print(f"--- raw query {i} " + "-" * 50)
    print(f"  calls:          {q.get('calls')}")
    print(f"  mean_exec_time: {q.get('mean_exec_time')} ms")
    print(f"  avg_rows:       {q.get('avg_rows_returned')}")
    print(f"  sql: {_short(q.get('query'))}")
    print()


---

# Step 2 of 7: Clean, repair, and describe with Bedrock

**This is the ~10-minute step.** It is also where the raw query log becomes a
training dataset.

For every extracted query, `clean_dataset` makes up to three round-trips:

1. **Placeholder substitution** — replace `$1`, `$2` with plausible literals, then
   **execute the query against Aurora** to confirm it still runs.
2. **Error-driven repair** — if execution failed, hand the SQL *and the Postgres
   error message* to Bedrock, ask for a fix, and execute again. One attempt.
3. **Description generation** — ask Bedrock to write the natural-language question
   this query answers. That question becomes the `user_query` the model will be
   trained to translate.

Two things are worth noticing about this design. First, **every surviving pair has
been verified by execution** — we never train on SQL we have not seen run. Second,
the expensive part is not the LLM calls; it is that each one is followed by a real
database round-trip. Verification costs more than generation.

Expect the count to drop. Queries that cannot be repaired into something
executable are dropped on purpose.

In [ ]:
# [Step 2/7] Clean, validate against the database, and describe with Bedrock.
# This is the slow step — roughly 10 minutes. Progress is logged per query.
logger.info("\n[Step 2/7] Cleaning and validating extracted queries...")
cleaned_samples = clean_dataset(raw_queries)

if len(cleaned_samples) < MIN_SAMPLES_REQUIRED:
    raise RuntimeError(
        f"Only {len(cleaned_samples)} samples survived cleaning; need at least "
        f"{MIN_SAMPLES_REQUIRED}. Consider lowering MIN_QUERY_LENGTH, or run more "
        "queries against the database to enrich pg_stat_statements."
    )

print(f"\n{len(raw_queries)} raw -> {len(cleaned_samples)} verified training pairs")
print(f"({len(raw_queries) - len(cleaned_samples)} dropped as unrepairable or out of bounds)")


In [ ]:
# Preview: your actual training pairs.
#
# This is the most important view in the lab. Each `user_query` was written by
# Bedrock; each `expected_sql` came out of your database's own query log and has
# been executed successfully. Read a couple and ask yourself: is this a question
# a person would plausibly ask, and does the SQL answer it?
#
# If the descriptions look generic or repetitive, that is a real signal — the
# model can only learn the mapping that is in the data.

print(f"{len(cleaned_samples)} cleaned samples. First 2:\n")
for i, s in enumerate(cleaned_samples[:2], 1):
    print(f"--- sample {i} " + "-" * 55)
    print(f"  user_query:   {s.get('user_query')}")
    print(f"  expected_sql: {_short(s.get('expected_sql'))}")
    print()


---

# Step 3 of 7: Extract the schema

The model needs to know the table and column names to generate SQL against them.
There are two ways to arrange that: hope it memorises the schema during training,
or **put the schema in the prompt.** We do the second, because it is far more
reliable and it degrades gracefully when the schema changes.

This is also the honest answer to "why not just use a frontier model?" — a
frontier model can absolutely write good SQL, but it has never seen your schema.
The schema context is how any model, fine-tuned or not, learns what
`product_sales` contains.


In [ ]:
# [Step 3/7] Extract schema context for prompt enrichment
logger.info("\n[Step 3/7] Extracting database schema context...")
schema_context = extract_schema_context()

if schema_context:
    print(f"Schema context extracted ({len(schema_context)} chars)")
else:
    print("No schema context available — prompts will be generic")


In [ ]:
# Preview: the schema block that goes into every training prompt.
print(_short(schema_context, 1200) if schema_context else "(empty)")


---

# Step 4 of 7: Format for SFT

Supervised fine-tuning wants the simplest possible shape: one input string, one
target string. `format_for_training` builds the prompt by wrapping the
`user_query` in an instruction template together with the schema block, and uses
the verified SQL as the completion.

The template matters more than it looks. Whatever wrapper you train with is the
wrapper you must use at inference time — if Lab 2's prompt format drifted from
this one, the model would be answering a question it was never asked. This is the
single most common cause of a fine-tuned model that "worked in training" and
fails in production.


In [ ]:
# [Step 4/7] Format samples for SageMaker AI SFT training
logger.info("\n[Step 4/7] Formatting data for SageMaker AI SFT training...")
formatted_samples = [
    format_for_training(sample, schema_context) for sample in cleaned_samples
]

print(f"Formatted {len(formatted_samples)} SFT records")


In [ ]:
# Preview: one complete SFT record, exactly as it will be written to JSONL.
#
# The prompt includes the schema block from Step 3. The completion is bare SQL
# with no explanation and no markdown fence — that formatting discipline is a
# large part of what SFT teaches, and it is why the Lab 3 ablation arm (which
# skips SFT) produces verbose output that the reward function has to strip.

_rec = formatted_samples[0]
print("prompt:")
print(_rec["prompt"])
print("completion:")
print(_short(_rec["completion"]))


---

# Step 5 of 7: Split into train and validation

85% training, 15% held out, shuffled with a fixed seed.

The validation set is how Lab 2 measures whether the model *generalized* rather
than memorized. Lab 2 and Lab 4 also evaluate against a **combined** set (train +
validation), which sounds like cheating but is not — it answers a different
question. Validation tells you how the model handles queries it has never seen;
combined tells you how it performs on the mix of traffic it will actually face,
where repeat queries are the majority. Both numbers are useful; reporting only
the second would be dishonest.


In [ ]:
# [Step 5/7] Split into train/validation and write the SFT JSONL files
logger.info("\n[Step 5/7] Splitting into train/validation sets...")
train_set, val_set = split_dataset(formatted_samples)

with open(TRAIN_OUTPUT, "w") as f:
    for record in train_set:
        f.write(json.dumps(record) + "\n")

with open(VALIDATION_OUTPUT, "w") as f:
    for record in val_set:
        f.write(json.dumps(record) + "\n")

# Combined = train + validation, used by Lab 2 and Lab 4 for the
# "performance on real traffic" view.
with open(COMBINED_OUTPUT, "w") as f:
    for record in formatted_samples:
        f.write(json.dumps(record) + "\n")

print(f"  {TRAIN_OUTPUT}:      {len(train_set)} records")
print(f"  {VALIDATION_OUTPUT}: {len(val_set)} records")
print(f"  {COMBINED_OUTPUT}:   {len(formatted_samples)} records")


---

# Step 6 of 7: Format for RLVR

Lab 3 needs the same data in a structurally different shape. Three differences
matter:

| | SFT | RLVR (VERL format) |
|---|---|---|
| Prompt | one instruction string | list of `{role, content}` messages |
| Target SQL | `completion` — shown to the model | `reward_model.ground_truth` — shown only to the reward function |
| Extras | none | `data_source`, `ability`, `extra_info.split` |

That middle row is the conceptual difference between the two training methods,
expressed as a data layout. In SFT the model is shown the right answer and
learns to reproduce it. In RLVR the model generates its own candidates and never
sees the ground truth at all — the ground truth goes to the evaluator, which
executes both queries and returns a score. The model learns from the *reward*,
not from the answer.

## One load-bearing detail

The cell below re-splits with `split_dataset(cleaned_samples)`. Because
`split_dataset` seeds `random` with `42` on every call, and `cleaned_samples` and
`formatted_samples` are the same length in the same order, **the RLVR split is
row-aligned with the SFT split** — RLVR's validation set contains exactly the
samples SFT held out.

This is not cosmetic. It is what makes Lab 4's comparison table meaningful: every
model in it is scored on the same held-out rows. Change the seed in one place and
the comparison quietly stops being like-for-like.

In [ ]:
# [Step 6/7] Format and write the RLVR datasets
logger.info("\n[Step 6/7] Formatting data for SageMaker AI RLVR training...")
os.makedirs(RLVR_DIR, exist_ok=True)

# Re-split the *cleaned* samples rather than the SFT-formatted ones.
#
# LOAD-BEARING: split_dataset() calls random.seed(42) on every invocation, and
# cleaned_samples is the same length and order as formatted_samples. So this
# produces the identical partition — the RLVR validation set holds exactly the
# rows the SFT validation set holds. Lab 4 compares SFT, RLVR and frontier
# models on the same held-out data, which only works because of this alignment.
# If you change the seed or reorder either list, change both.
train_samples_raw, val_samples_raw = split_dataset(cleaned_samples)

with open(RLVR_TRAIN_OUTPUT, "w") as f:
    for i, sample in enumerate(train_samples_raw):
        record = format_for_rlvr(sample, schema_context, index=i)
        record["extra_info"]["split"] = "train"
        f.write(json.dumps(record) + "\n")

with open(RLVR_VAL_OUTPUT, "w") as f:
    for i, sample in enumerate(val_samples_raw):
        record = format_for_rlvr(sample, schema_context, index=i)
        record["extra_info"]["split"] = "val"
        f.write(json.dumps(record) + "\n")

with open(RLVR_COMBINED_OUTPUT, "w") as f:
    for i, sample in enumerate(cleaned_samples):
        record = format_for_rlvr(sample, schema_context, index=i)
        record["extra_info"]["split"] = "combined"
        f.write(json.dumps(record) + "\n")

print(f"  {RLVR_TRAIN_OUTPUT}:    {len(train_samples_raw)} records")
print(f"  {RLVR_VAL_OUTPUT}:      {len(val_samples_raw)} records")
print(f"  {RLVR_COMBINED_OUTPUT}: {len(cleaned_samples)} records")


In [ ]:
# Preview: two complete RLVR records, before they go anywhere near SageMaker AI.
#
# Compare against the SFT record in Step 4. The prompt is now a message list
# with the schema in the `system` role; the SQL has moved to
# reward_model.ground_truth where only the evaluator will see it.

with open(RLVR_VAL_OUTPUT) as f:
    _rlvr_preview = [json.loads(line) for line in f if line.strip()][:2]

for i, rec in enumerate(_rlvr_preview, 1):
    print(f"--- RLVR record {i} " + "-" * 48)
    # Truncate the system message (it contains the full schema block) so the
    # structure stays readable.
    display_rec = json.loads(json.dumps(rec))
    for msg in display_rec["prompt"]:
        if msg["role"] == "system":
            msg["content"] = _short(msg["content"], 200)
    display_rec["reward_model"]["ground_truth"] = _short(
        display_rec["reward_model"]["ground_truth"], 200
    )
    print(json.dumps(display_rec, indent=2))
    print()


---

# Step 7 of 7: Validate and profile

Six files are on disk. Before Lab 2 and Lab 3 train on them, confirm they are
well-formed and take a look at what is actually in them.

Two validators run, because the two formats have nothing structurally in common:

- `validate_jsonl_file` checks the SFT shape — non-empty `prompt` and `completion`.
- `validate_rlvr_jsonl_file` (defined below) checks the RLVR shape — `prompt` as a
  non-empty message list, and a non-empty `reward_model.ground_truth`.

The second one is worth having. A malformed RLVR file does not fail loudly at
upload time; it fails an hour into a GPU training job in Lab 3, which is an
expensive place to discover a missing field.

`run_quality_checks` then profiles the dataset: SQL complexity distribution and
the duplicate-description ratio. Read the issues list if it prints anything.


In [ ]:
# [Step 7/7] Validate both formats and profile the dataset
logger.info("\n[Step 7/7] Running final validation and quality checks...")


def validate_rlvr_jsonl_file(filepath: str) -> bool:
    """Validate the RLVR (VERL) record shape.

    validate_jsonl_file() checks for prompt/completion, which is the SFT shape —
    RLVR records have neither field, so they need their own structural check.
    A missing field here surfaces as a failed GPU training job in Lab 3, so it
    is worth catching now.
    """
    logger.info(f"Validating {filepath} (RLVR format)...")
    errors, line_count = [], 0

    with open(filepath) as f:
        for i, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError as e:
                errors.append(f"Line {i}: invalid JSON - {e}")
                continue

            prompt = record.get("prompt")
            if not isinstance(prompt, list) or not prompt:
                errors.append(f"Line {i}: 'prompt' must be a non-empty list of messages")
            else:
                for j, msg in enumerate(prompt):
                    if not isinstance(msg, dict) or "role" not in msg or "content" not in msg:
                        errors.append(f"Line {i}: message {j} missing 'role' or 'content'")
                    elif not str(msg["content"]).strip():
                        errors.append(f"Line {i}: message {j} has empty content")

            ground_truth = record.get("reward_model", {}).get("ground_truth", "")
            if not str(ground_truth).strip():
                errors.append(f"Line {i}: missing or empty reward_model.ground_truth")

            line_count += 1

    if errors:
        for err in errors[:10]:
            logger.error(err)
        logger.error(f"Validation FAILED: {len(errors)} errors in {filepath}")
        return False

    logger.info(f"Validation PASSED: {line_count} valid records in {filepath}")
    return True


sft_valid = all(
    validate_jsonl_file(f) for f in (TRAIN_OUTPUT, VALIDATION_OUTPUT, COMBINED_OUTPUT)
)
rlvr_valid = all(
    validate_rlvr_jsonl_file(f)
    for f in (RLVR_TRAIN_OUTPUT, RLVR_VAL_OUTPUT, RLVR_COMBINED_OUTPUT)
)

quality_stats = run_quality_checks(cleaned_samples)


In [ ]:
# Pipeline summary — read the counts back off disk, not out of memory.
#
# Counting lines in the written files (rather than reusing the in-memory lists)
# is the last chance to catch a truncated or half-written file before the AI
# Registry uploads it.

_files = {
    "SFT train":      TRAIN_OUTPUT,
    "SFT validation": VALIDATION_OUTPUT,
    "SFT combined":   COMBINED_OUTPUT,
    "RLVR train":     RLVR_TRAIN_OUTPUT,
    "RLVR validation": RLVR_VAL_OUTPUT,
    "RLVR combined":  RLVR_COMBINED_OUTPUT,
}

print("=" * 70)
print("PIPELINE SUMMARY")
print("=" * 70)
print(f"  Raw queries extracted:    {len(raw_queries)}")
print(f"  After cleaning:           {len(cleaned_samples)}")
print()
print("  Files on disk:")
missing = []
for label, path in _files.items():
    if not os.path.exists(path):
        missing.append(path)
        print(f"    {label:18s} MISSING  {path}")
        continue
    with open(path) as f:
        n = sum(1 for line in f if line.strip())
    print(f"    {label:18s} {n:4d} records  {path}")

print()
print(f"  Avg SQL length:           {quality_stats['avg_query_length']} chars")
print(f"  Avg description length:   {quality_stats['avg_description_length']} chars")
print(f"  Unique descriptions:      {quality_stats['unique_description_count']} / {len(cleaned_samples)}")
print(f"  Queries with GROUP BY:    {quality_stats['has_group_by']}")
print(f"  Queries with JOIN:        {quality_stats['has_join']}")
print(f"  Queries with WHERE:       {quality_stats['has_where']}")
print(f"  Queries with subqueries:  {quality_stats['has_subquery']}")
print()
print(f"  SFT JSONL valid:          {'PASS' if sft_valid else 'FAIL'}")
print(f"  RLVR JSONL valid:         {'PASS' if rlvr_valid else 'FAIL'}")

if quality_stats["issues"]:
    print("\n  Quality issues:")
    for issue in quality_stats["issues"]:
        print(f"    {issue}")

print("=" * 70)
if sft_valid and rlvr_valid and not missing:
    print("All six datasets ready. Register them in the AI Registry below.")
else:
    print("Problems found above — resolve them before registering the datasets.")


## Register Datasets in SageMaker AI Registry

We register each dataset as a versioned artifact in the SageMaker AI Registry. This gives us reproducibility (every training job references a specific dataset version by ARN) and makes it easy to swap datasets without changing notebook code.

In [ ]:
from sagemaker.ai_registry.dataset import DataSet, CustomizationTechnique

# Dataset names — these are the registry lookup keys. Later labs use
# DataSet.get(<name>) to resolve a dataset back to its S3 source, so the
# names are persisted alongside the ARNs at the end of this notebook.
TRAINING_DATASET_NAME = "sft-dataset-training"
VALIDATION_DATASET_NAME = "sft-dataset-validation"
SFT_COMBINED_DATASET_NAME = "sft-dataset-combined"
RLVR_TRAINING_DATASET_NAME = "rlvr-dataset-training"
RLVR_VALIDATION_DATASET_NAME = "rlvr-dataset-validation"
RLVR_COMBINED_DATASET_NAME = "rlvr-dataset-combined"

# customization_technique tags each dataset with the method it is meant for —
# CustomizationTechnique.SFT for the supervised sets, .RLVR for the
# reinforcement sets. It is optional (the registration works without it), but
# tagging it makes each dataset self-describing in the AI Registry and lets the
# Lab 2/3 trainers filter for datasets that match their customization method.
# wait=True blocks until each dataset reaches AVAILABLE, so the ARNs are
# resolvable by the time Lab 2 looks them up.
sft_training_dataset = DataSet.create(
    name=TRAINING_DATASET_NAME,
    source="train.jsonl",
    customization_technique=CustomizationTechnique.SFT,
    wait=True
)

sft_validation_dataset = DataSet.create(
    name=VALIDATION_DATASET_NAME,
    source="validation.jsonl",
    customization_technique=CustomizationTechnique.SFT,
    wait=True
)

sft_combined_dataset = DataSet.create(
    name=SFT_COMBINED_DATASET_NAME,
    source="combined.jsonl",
    customization_technique=CustomizationTechnique.SFT,
    wait=True
)

rlvr_training_dataset = DataSet.create(
    name=RLVR_TRAINING_DATASET_NAME,
    source="rlvr_data/rl_train.jsonl",
    customization_technique=CustomizationTechnique.RLVR,
    wait=True
)

rlvr_validation_dataset = DataSet.create(
    name=RLVR_VALIDATION_DATASET_NAME,
    source="rlvr_data/rl_val.jsonl",
    customization_technique=CustomizationTechnique.RLVR,
    wait=True
)

rlvr_combined_dataset = DataSet.create(
    name=RLVR_COMBINED_DATASET_NAME,
    source="rlvr_data/rl_combined.jsonl",
    customization_technique=CustomizationTechnique.RLVR,
    wait=True
)

print(f"SFT Training:      {sft_training_dataset.arn}")
print(f"SFT Validation:    {sft_validation_dataset.arn}")
print(f"SFT Combined:      {sft_combined_dataset.arn}")
print(f"RLVR Training:     {rlvr_training_dataset.arn}")
print(f"RLVR Validation:   {rlvr_validation_dataset.arn}")
print(f"RLVR Combined:     {rlvr_combined_dataset.arn}")

In [ ]:
# Persist dataset ARNs and names for subsequent labs
TRAINING_DATASET_ARN = sft_training_dataset.arn
VALIDATION_DATASET_ARN = sft_validation_dataset.arn
SFT_COMBINED_DATASET_ARN = sft_combined_dataset.arn
RLVR_TRAINING_DATASET_ARN = rlvr_training_dataset.arn
RLVR_VALIDATION_DATASET_ARN = rlvr_validation_dataset.arn
RLVR_COMBINED_DATASET_ARN = rlvr_combined_dataset.arn
%store TRAINING_DATASET_ARN VALIDATION_DATASET_ARN SFT_COMBINED_DATASET_ARN
%store RLVR_TRAINING_DATASET_ARN RLVR_VALIDATION_DATASET_ARN RLVR_COMBINED_DATASET_ARN
%store TRAINING_DATASET_NAME VALIDATION_DATASET_NAME SFT_COMBINED_DATASET_NAME
%store RLVR_TRAINING_DATASET_NAME RLVR_VALIDATION_DATASET_NAME RLVR_COMBINED_DATASET_NAME